# NLP Early Warning Framework

## Overview

This notebook builds a rule-based monthly watchlist from complaint text and cohort trends.
It is not a supervised prediction model.

The goal is to help identify make/model/component cohorts that look like emerging issues based on:
- complaint count growth
- unusual month-over-month behavior compared with prior months
- broad severity share
- complaint text clues

Inputs:
- `data/processed/odi_component_multilabel_cases.parquet`
- `data/processed/odi_component_text_sidecar.parquet`

Outputs:
- `data/outputs/nlp_early_warning_watchlist.csv`
- `data/outputs/nlp_early_warning_terms.csv`

## Setup

This cell sets imports, project paths, and helper constants.
You should be able to run this notebook from VS Code or Colab as long as the repo files are available.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

RANDOM_SEED = 42

# Find repo root dynamically so notebook works in local/remote environments
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUTS_DIR = PROJECT_ROOT / 'data' / 'outputs'

MULTI_PATH = PROCESSED_DATA_DIR / 'odi_component_multilabel_cases.parquet'
SIDECAR_PATH = PROCESSED_DATA_DIR / 'odi_component_text_sidecar.parquet'
WATCHLIST_OUTPUT_PATH = OUTPUTS_DIR / 'nlp_early_warning_watchlist.csv'
TERMS_OUTPUT_PATH = OUTPUTS_DIR / 'nlp_early_warning_terms.csv'

print(f"Project root: {PROJECT_ROOT}")
print(f"Multi-label input: {MULTI_PATH}")
print(f"Text sidecar input: {SIDECAR_PATH}")
print(f"Watchlist output: {WATCHLIST_OUTPUT_PATH}")
print(f"Terms output: {TERMS_OUTPUT_PATH}")

## Data Loading

Load the two processed datasets and join them on `odino`.
The text sidecar gives one cleaned text row per complaint case.

In [ ]:
if not MULTI_PATH.exists():
    raise FileNotFoundError(f"Missing input file: {MULTI_PATH}")
if not SIDECAR_PATH.exists():
    raise FileNotFoundError(f"Missing input file: {SIDECAR_PATH}")

multi_df = pd.read_parquet(MULTI_PATH)
sidecar_df = pd.read_parquet(SIDECAR_PATH)

join_cols = [
    'odino',
    'cdescr',
    'cdescr_model_text',
    'cdescr_missing_flag',
    'cdescr_placeholder_flag',
    'cdescr_char_len',
    'cdescr_word_count'
]
sidecar_use = sidecar_df[join_cols].copy()

base_df = multi_df.merge(sidecar_use, on='odino', how='left', validate='one_to_one')
base_df['ldate'] = pd.to_datetime(base_df['ldate'])
base_df['month'] = base_df['ldate'].dt.to_period('M')

text_candidate = base_df['cdescr_model_text'].fillna('').astype(str).str.strip()
fallback_text = base_df['cdescr'].fillna('').astype(str).str.strip()
base_df['text_for_nlp'] = np.where(text_candidate.ne(''), text_candidate, fallback_text)

print(f"Base rows: {len(base_df):,}")
print(f"Date range: {base_df['ldate'].min().date()} to {base_df['ldate'].max().date()}")
base_df[['odino', 'month', 'mfr_name', 'maketxt', 'modeltxt', 'component_groups', 'text_for_nlp']].head(3)

## Cohort Table

Create complaint-component rows by exploding `component_groups`.
Then aggregate to monthly cohort rows by:
- `mfr_name`
- `maketxt`
- `modeltxt`
- `yeartxt`
- `component_group`
- `month`

In [ ]:
# TODO: Choose the cohort definition. Good starter value: ['mfr_name', 'maketxt', 'modeltxt', 'yeartxt', 'component_group']
cohort_key_cols = None

if cohort_key_cols is None:
    raise ValueError('TODO: Set cohort_key_cols before building monthly cohorts.')


rows_df = base_df.copy()
rows_df['component_group'] = (
    rows_df['component_groups']
    .fillna('')
    .astype(str)
    .str.split('|')
)
rows_df = rows_df.explode('component_group', ignore_index=True)
rows_df['component_group'] = rows_df['component_group'].fillna('').astype(str).str.strip()
rows_df = rows_df.loc[rows_df['component_group'].ne('')].copy()

monthly_df = (
    rows_df
    .groupby(cohort_key_cols + ['month'], as_index=False)
    .agg(
        complaint_count=('odino', 'nunique'),
        broad_severity_count=('severity_broad_flag', 'sum'),
        primary_severity_count=('severity_primary_flag', 'sum'),
        unique_states=('state', 'nunique'),
        avg_text_len=('cdescr_char_len', 'mean')
    )
)

monthly_df['broad_severity_rate'] = monthly_df['broad_severity_count'] / monthly_df['complaint_count'].clip(lower=1)
monthly_df['primary_severity_rate'] = monthly_df['primary_severity_count'] / monthly_df['complaint_count'].clip(lower=1)

monthly_df = monthly_df.sort_values(cohort_key_cols + ['month']).reset_index(drop=True)

print(f"Exploded complaint-component rows: {len(rows_df):,}")
print(f"Monthly cohort rows: {len(monthly_df):,}")
monthly_df.head(5)

## Leakage-Safe Features

Build prior/rolling features using only earlier months for each cohort.
No future month values are used to build the current month score.

Features added:
- `prior_3m_mean_count`
- `prior_6m_mean_count`
- `prior_6m_std_count`
- `growth_ratio`
- `z_score`


In [ ]:
# TODO: Choose how many earlier months should define recent and longer-history baselines. Good starter values are 3 and 6.
SHORT_WINDOW_MONTHS = None
LONG_WINDOW_MONTHS = None

if SHORT_WINDOW_MONTHS is None or LONG_WINDOW_MONTHS is None:
    raise ValueError('TODO: Set SHORT_WINDOW_MONTHS and LONG_WINDOW_MONTHS before creating prior-month features.')
if SHORT_WINDOW_MONTHS <= 0 or LONG_WINDOW_MONTHS <= 0:
    raise ValueError('Window sizes must be positive integers.')

feature_df = monthly_df.sort_values(cohort_key_cols + ['month']).reset_index(drop=True).copy()

# Each prior feature below uses only earlier rows within the same cohort.
cohort_grouped = feature_df.groupby(cohort_key_cols, sort=False, dropna=False)
feature_df['months_with_history'] = cohort_grouped.cumcount()

max_window = max(SHORT_WINDOW_MONTHS, LONG_WINDOW_MONTHS)
lag_cols = []
for lag in range(1, max_window + 1):
    col = f'_prior_count_lag_{lag}'
    feature_df[col] = cohort_grouped['complaint_count'].shift(lag)
    lag_cols.append(col)

short_lag_cols = lag_cols[:SHORT_WINDOW_MONTHS]
long_lag_cols = lag_cols[:LONG_WINDOW_MONTHS]

feature_df['prior_3m_mean_count'] = feature_df[short_lag_cols].mean(axis=1, skipna=True)
feature_df['prior_6m_mean_count'] = feature_df[long_lag_cols].mean(axis=1, skipna=True)
prior_6m_non_missing = feature_df[long_lag_cols].notna().sum(axis=1)
feature_df['prior_6m_std_count'] = feature_df[long_lag_cols].std(axis=1, skipna=True, ddof=0).where(prior_6m_non_missing >= 2)

feature_df['growth_ratio'] = feature_df['complaint_count'] / feature_df['prior_6m_mean_count'].replace(0, np.nan)
denom = feature_df['prior_6m_std_count'].replace(0, np.nan)
feature_df['z_score'] = (feature_df['complaint_count'] - feature_df['prior_6m_mean_count']) / denom
feature_df['z_score'] = feature_df['z_score'].replace([np.inf, -np.inf], np.nan)
feature_df = feature_df.drop(columns=lag_cols)

feature_df[['complaint_count', 'prior_3m_mean_count', 'prior_6m_mean_count', 'prior_6m_std_count', 'growth_ratio', 'z_score']].describe().T


## Watchlist Score

Create a fixed rule-based score using:
- positive z-score
- positive growth ratio above 1.0
- complaint volume
- broad severity rate

Then keep rows where:
- at least one prior month exists (`months_with_history >= 1`)
- min complaints and top kept complaints >= set values

In [ ]:
# TODO: Choose watchlist settings. Good starter values are shown in the comments below.
WATCHLIST_WEIGHTS = None  # Try {'z': 0.35, 'growth': 0.30, 'volume': 0.20, 'severity': 0.15} and adjust based on results
MIN_COMPLAINTS_PER_COHORT_MONTH = None  # Try 3 and adjust based on resulting watchlist sizes
TOP_N_PER_MONTH = None  # Try 25 and adjust based on resulting watchlist sizes

if WATCHLIST_WEIGHTS is None or MIN_COMPLAINTS_PER_COHORT_MONTH is None or TOP_N_PER_MONTH is None:
    raise ValueError('TODO: Fill in WATCHLIST_WEIGHTS, MIN_COMPLAINTS_PER_COHORT_MONTH, and TOP_N_PER_MONTH before scoring the watchlist.')

required_weight_keys = {'z', 'growth', 'volume', 'severity'}
if set(WATCHLIST_WEIGHTS) != required_weight_keys:
    raise ValueError(f'WATCHLIST_WEIGHTS must contain exactly these keys: {sorted(required_weight_keys)}')

score_df = feature_df.copy()

score_df['z_score_pos'] = score_df['z_score'].clip(lower=0)
score_df['growth_ratio_pos'] = (score_df['growth_ratio'] - 1.0).clip(lower=0)
score_df['volume_score_raw'] = np.log1p(score_df['complaint_count'])
score_df['severity_score_raw'] = score_df['broad_severity_rate'].fillna(0)

z_scale = score_df['z_score_pos'].quantile(0.95)
g_scale = score_df['growth_ratio_pos'].quantile(0.95)
v_scale = score_df['volume_score_raw'].max()

score_df['z_score_norm'] = score_df['z_score_pos'] / z_scale if pd.notna(z_scale) and z_scale > 0 else 0
score_df['growth_ratio_norm'] = score_df['growth_ratio_pos'] / g_scale if pd.notna(g_scale) and g_scale > 0 else 0
score_df['volume_norm'] = score_df['volume_score_raw'] / v_scale if pd.notna(v_scale) and v_scale > 0 else 0
score_df['severity_norm'] = score_df['severity_score_raw'].clip(lower=0, upper=1)

for col in ['z_score_norm', 'growth_ratio_norm', 'volume_norm', 'severity_norm']:
    score_df[col] = score_df[col].clip(lower=0, upper=1)

score_df['watchlist_score'] = (
    WATCHLIST_WEIGHTS['z'] * score_df['z_score_norm'] +
    WATCHLIST_WEIGHTS['growth'] * score_df['growth_ratio_norm'] +
    WATCHLIST_WEIGHTS['volume'] * score_df['volume_norm'] +
    WATCHLIST_WEIGHTS['severity'] * score_df['severity_norm']
)

filtered_df = score_df.loc[
    (score_df['complaint_count'] >= MIN_COMPLAINTS_PER_COHORT_MONTH) &
    (score_df['months_with_history'] >= 1)
].copy()

filtered_df['month_rank'] = (
    filtered_df
    .groupby('month')['watchlist_score']
    .rank(method='first', ascending=False)
)

watchlist_df = (
    filtered_df
    .loc[filtered_df['month_rank'] <= TOP_N_PER_MONTH]
    .sort_values(['month', 'month_rank', 'watchlist_score'], ascending=[True, True, False])
    .reset_index(drop=True)
)

WATCHLIST_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
watchlist_df.to_csv(WATCHLIST_OUTPUT_PATH, index=False)

print(f"Watchlist rows: {len(watchlist_df):,}")
print(f"Saved: {WATCHLIST_OUTPUT_PATH}")
watchlist_df.head(10)

## Text Clues

For top watchlist cohort-month rows, extract:
- a few sample complaint narratives
- top TF-IDF terms/phrases

These are clues for review.

In [ ]:
latest_month = watchlist_df['month'].max() if len(watchlist_df) else None
if latest_month is None:
    selected_watchlist = watchlist_df.copy()
else:
    selected_watchlist = watchlist_df.loc[watchlist_df['month'].eq(latest_month)].sort_values('month_rank').head(5).copy()

display_cols = ['month', 'month_rank', *cohort_key_cols, 'complaint_count', 'watchlist_score']
missing_display_cols = [col for col in display_cols if col not in selected_watchlist.columns]
if missing_display_cols:
    raise KeyError(f'Missing expected watchlist columns: {missing_display_cols}')

selected_watchlist[display_cols].head(10)


In [ ]:
def top_terms_for_docs(text_series: pd.Series, top_n: int = 10) -> pd.DataFrame:
    docs = text_series.fillna('').astype(str).str.strip()
    docs = docs.loc[docs.ne('')]
    if docs.empty:
        return pd.DataFrame(columns=['term', 'tfidf_score'])

    vectorizer = TfidfVectorizer(
        lowercase=True,
        analyzer='word',
        ngram_range=(1, 2),
        min_df=1,
        max_features=2000
    )
    mat = vectorizer.fit_transform(docs)
    avg_scores = np.asarray(mat.mean(axis=0)).ravel()
    terms = np.array(vectorizer.get_feature_names_out())
    top_idx = np.argsort(avg_scores)[::-1][:top_n]
    term_df = pd.DataFrame({'term': terms[top_idx], 'tfidf_score': avg_scores[top_idx]})
    return term_df

term_rows = []
example_rows = []

for _, row in selected_watchlist.iterrows():
    mask = rows_df['month'].eq(row['month'])
    for col in cohort_key_cols:
        mask &= rows_df[col].eq(row[col])
    cohort_rows = rows_df.loc[mask].copy()

    cohort_identity = {col: row[col] for col in cohort_key_cols}

    sample_text = (
        cohort_rows['text_for_nlp']
        .fillna('')
        .astype(str)
        .str.strip()
        .loc[lambda s: s.ne('')]
        .drop_duplicates()
        .head(3)
    )
    for idx, txt in enumerate(sample_text, start=1):
        example_rows.append({
            'month': str(row['month']),
            **cohort_identity,
            'example_index': idx,
            'example_text': txt
        })

    terms_df = top_terms_for_docs(cohort_rows['text_for_nlp'], top_n=10)
    for _, trow in terms_df.iterrows():
        term_rows.append({
            'month': str(row['month']),
            **cohort_identity,
            'watchlist_score': row['watchlist_score'],
            'term': trow['term'],
            'tfidf_score': float(trow['tfidf_score'])
        })

terms_df = pd.DataFrame(term_rows)
examples_df = pd.DataFrame(example_rows)

TERMS_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
terms_df.to_csv(TERMS_OUTPUT_PATH, index=False)

print(f"Term rows: {len(terms_df):,}")
print(f"Saved: {TERMS_OUTPUT_PATH}")
terms_df.head(15)


## Results

Quick summary tables for monthly watchlist outputs and text examples.

In [ ]:
monthly_summary = (
    watchlist_df
    .groupby('month', as_index=False)
    .agg(
        cohort_rows=('watchlist_score', 'count'),
        avg_watchlist_score=('watchlist_score', 'mean'),
        max_watchlist_score=('watchlist_score', 'max'),
        avg_complaint_count=('complaint_count', 'mean')
    )
)

print('Monthly watchlist summary:')
display(monthly_summary.tail(12))

print('Example complaint text snippets for selected cohorts:')
display(examples_df.head(15))

## Next TODOs / Extension Ideas

Use this section as a checklist for improving the early-warning notebook after the starter watchlist runs.

Suggested next steps:
- Change `cohort_key_cols` and compare how the watchlist changes. A more specific cohort like make/model/year/component is precise but can be noisy; a broader cohort like make/model/component is more stable.
- Try different `SHORT_WINDOW_MONTHS` and `LONG_WINDOW_MONTHS` values. Short windows react faster, while longer windows are less sensitive to one unusual month.
- Tune `WATCHLIST_WEIGHTS`, `MIN_COMPLAINTS_PER_COHORT_MONTH`, and `TOP_N_PER_MONTH` so the output is small enough for a person to review.
- Add a repeated-signal summary that counts which cohorts appear in the monthly top list more than once.
- Add a chart of complaint counts over time for a selected cohort so the score can be checked visually.
- Add more text clues, such as the top repeated phrases for each selected cohort or a few representative complaint narratives.
- Add a manual review column to the saved CSV, such as `review_notes` or `looks_actionable`, if the team wants to inspect rows together.
- Document false positives. If a row looks like noise, write down why so the score can be improved later.

A high watchlist score means a cohort deserves a closer look.